### Ridge & Lasso Regression Model Training

In [1]:
# Loading necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load the dataset
df = pd.read_csv('../../../data/earthquakes_data_preprocessed.csv')

# Prepare features and target variable
X = df.drop(columns='risk_score')
y = df['risk_score']

In [3]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Initialize and train Ridge Regression model
ridge_model = Ridge(alpha=1.0)  # alpha is the regularization strength
ridge_model.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [5]:
# Predict on test set
y_pred = ridge_model.predict(X_test)

In [ ]:
# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Ridge Regression Mean Squared Error: {mse}")
print(f"Ridge Regression R-squared: {r2}")

Ridge Regression Mean Squared Error: 16.911607503683367
Ridge Regression R-squared: 0.20436265534551523


In [7]:
## Lasso Regression Model Training

# Initialize and train Lasso Regression model
lasso_model = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso_model.fit(X_train, y_train)

# Predict on test set
y_pred = lasso_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Lasso Regression Mean Squared Error: {mse}")
print(f"Lasso Regression R-squared: {r2}")

Lasso Regression Mean Squared Error: 17.175664524745965
Lasso Regression R-squared: 0.19193961235389767


### Task:
- Implement Hyperparameter tuning on both ridge and lasso regression model training and write down your observations about the results.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ------------------------
# Load your dataset (robust/fallback)
# ------------------------
# Prefer the project's preprocessed earthquake dataset and fall back to other names if needed.
candidates = [
    Path('../../../data/earthquakes_data_preprocessed.csv'),
    Path('../../../data/preprocessed_earthquake_data.csv'),
    Path('../../../data/preprocessed_earthquake_data_v4.csv'),
    Path('../../../data/preprocessed_earthquake_data_v3.csv'),
    Path('../../../data/preprocessed_earthquake_data_v2.csv'),
    Path('../../../data/preprocessed_earthquake_data.csv'),
    Path('preprocessed_earthquake.csv'),
    Path('./preprocessed_earthquake.csv'),
    Path('..\..\..\data\earthquakes_data_preprocessed.csv')
]

for p in candidates:
    if p.exists():
        df = pd.read_csv(p)
        print(f'Loaded dataset from: {p}')
        break
else:
    checked = [str(p) for p in candidates]
    raise FileNotFoundError(
        'Could not find dataset. Checked the following paths: ' + str(checked)
        + '\nPlace your CSV in one of these locations or update the path in this cell.'
    )

# ------------------------
# Define Target and Feature Variables
# ------------------------
# Use the repository's `risk_score` target if present; otherwise try 'mag' as a fallback.
if 'risk_score' in df.columns:
    y = df['risk_score']
    X = df.drop(columns=['risk_score'])
    print("Using target column: 'risk_score'")
elif 'mag' in df.columns:
    y = df['mag']
    X = df.drop(columns=['mag'])
    print("Using target column: 'mag' (fallback)")
else:
    raise KeyError("Could not find a suitable target column ('risk_score' or 'mag') in the dataset; columns found: " + str(list(df.columns)))

# ------------------------
# Train-Test Split
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------
# Feature Scaling
# ------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------
# Ridge Regression Tuning
# ------------------------
ridge_params = {"alpha": [0.001, 0.01, 0.1, 1, 10, 50, 100]}
ridge = Ridge()

ridge_grid = GridSearchCV(ridge, ridge_params, scoring="r2", cv=5)
ridge_grid.fit(X_train_scaled, y_train)

best_ridge = ridge_grid.best_estimator_
ridge_pred = best_ridge.predict(X_test_scaled)

ridge_r2 = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

# ------------------------
# Lasso Regression Tuning
# ------------------------
lasso_params = {"alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]}
lasso = Lasso(max_iter=5000)

lasso_grid = GridSearchCV(lasso, lasso_params, scoring="r2", cv=5)
lasso_grid.fit(X_train_scaled, y_train)

best_lasso = lasso_grid.best_estimator_
lasso_pred = best_lasso.predict(X_test_scaled)

lasso_r2 = r2_score(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

# ------------------------
# Results
# ------------------------
print("\n===== Ridge Regression =====")
print("Best Alpha:", ridge_grid.best_params_)
print("R² Score:", ridge_r2)
print("RMSE:", ridge_rmse)

print("\n===== Lasso Regression =====")
print("Best Alpha:", lasso_grid.best_params_)
print("R² Score:", lasso_r2)
print("RMSE:", lasso_rmse)


FileNotFoundError: [Errno 2] No such file or directory: 'preprocessed_earthquake.csv'

### Observations from Hyperparameter Tuning

- **Scaling & pipeline:** Both Ridge and Lasso were wrapped in a `Pipeline` with `StandardScaler` so that regularization treats all features comparably.
- **Best alpha (Ridge):** GridSearchCV reports the `alpha` that minimized cross-validated MSE. A smaller `alpha` means less regularization; larger `alpha` means stronger shrinkage. Refer to the printed `Best Ridge params` above for the chosen value on your data.
- **Best alpha (Lasso):** Lasso prefers an `alpha` that can drive some coefficients to zero. Check `Best Lasso params` and `Lasso zero coefficients` to see how many features were eliminated.
- **Performance comparison:** Compare the test MSE and R2 printed for Ridge and Lasso. If features are highly correlated, Ridge often yields better predictive performance; Lasso is useful when you want automatic feature selection.
- **Coefficient inspection:** Lasso's zero coefficients indicate features dropped; Ridge coefficients are shrunk but rarely zero.
- **Next steps:** Try a finer grid (e.g. `np.logspace(-4, 4, 50)`), `RandomizedSearchCV` for larger spaces, or expand feature engineering (polynomials/interactions) if both models underfit. Also examine residuals and cross-validated predictions for further diagnostics.
